# Pandas for Protein Data Analysis

This notebook introduces Pandas for handling tabular protein data.

**Learning Objectives:**
- Load and explore protein datasets
- Filter and select data efficiently
- Compute sequence statistics
- Prepare data for machine learning

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 50)

print(f"Pandas version: {pd.__version__}")

## 1. Creating a Sample Protein Dataset

We'll create a synthetic dataset similar to real protein property datasets.

In [ ]:
# Generate random protein sequences
np.random.seed(42)
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

def random_sequence(length):
    return ''.join(np.random.choice(list(amino_acids), length))

# Create dataset
n_samples = 1000
data = {
    'protein_id': [f'PROT{i:04d}' for i in range(n_samples)],
    'sequence': [random_sequence(np.random.randint(50, 500)) for _ in range(n_samples)],
    'organism': np.random.choice(['Human', 'Mouse', 'E.coli', 'Yeast'], n_samples),
    'soluble': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
    'thermal_stability': np.random.normal(50, 15, n_samples),  # Tm in Celsius
}

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Basic Data Exploration

In [ ]:
# Basic info
print("Dataset Info:")
print(df.info())

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Add sequence length column
df['length'] = df['sequence'].str.len()

# Length statistics
print(f"Length range: {df['length'].min()} - {df['length'].max()}")
print(f"Mean length: {df['length'].mean():.1f}")
print(f"Median length: {df['length'].median():.1f}")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Length distribution
axes[0].hist(df['length'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Count')
axes[0].set_title('Sequence Length Distribution')

# Solubility distribution
df['soluble'].value_counts().plot(kind='bar', ax=axes[1], color=['salmon', 'lightgreen'])
axes[1].set_xlabel('Soluble')
axes[1].set_ylabel('Count')
axes[1].set_title('Solubility Distribution')
axes[1].set_xticklabels(['Insoluble', 'Soluble'], rotation=0)

# Organism distribution
df['organism'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_xlabel('Organism')
axes[2].set_ylabel('Count')
axes[2].set_title('Organism Distribution')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Filtering and Selection

In [ ]:
# Filter by length
short_proteins = df[df['length'] < 100]
print(f"Short proteins (< 100 aa): {len(short_proteins)}")

# Filter by multiple conditions
filtered = df[(df['length'] >= 100) & (df['length'] <= 300) & (df['soluble'] == 1)]
print(f"Medium-length soluble proteins: {len(filtered)}")

# Filter by organism
human = df[df['organism'] == 'Human']
print(f"Human proteins: {len(human)}")

In [ ]:
# Select specific columns
subset = df[['protein_id', 'sequence', 'length', 'soluble']]
subset.head()

In [ ]:
# Using query method (more readable for complex conditions)
result = df.query('length > 200 and organism == "Human" and soluble == 1')
print(f"Long soluble human proteins: {len(result)}")
result.head()

## 4. Amino Acid Composition Analysis

In [ ]:
def compute_aa_composition(sequence):
    """
    Compute amino acid composition (frequency) of a sequence.
    
    Returns:
        Dictionary with amino acid frequencies
    """
    from collections import Counter
    counts = Counter(sequence)
    total = len(sequence)
    return {aa: counts.get(aa, 0) / total for aa in amino_acids}

# Test on one sequence
test_seq = df['sequence'].iloc[0]
comp = compute_aa_composition(test_seq)
print("Sample composition:")
for aa, freq in sorted(comp.items(), key=lambda x: -x[1])[:5]:
    print(f"  {aa}: {freq:.3f}")

In [ ]:
# Apply to all sequences
compositions = df['sequence'].apply(compute_aa_composition)
aa_df = pd.DataFrame(compositions.tolist())

# Add to main dataframe
df_with_aa = pd.concat([df, aa_df], axis=1)
print(f"New dataframe shape: {df_with_aa.shape}")
df_with_aa.head()

In [ ]:
# Compare AA composition between soluble and insoluble proteins
soluble_mean = df_with_aa[df_with_aa['soluble'] == 1][list(amino_acids)].mean()
insoluble_mean = df_with_aa[df_with_aa['soluble'] == 0][list(amino_acids)].mean()

diff = soluble_mean - insoluble_mean

plt.figure(figsize=(12, 5))
colors = ['green' if d > 0 else 'red' for d in diff]
plt.bar(diff.index, diff.values, color=colors, alpha=0.7)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Amino Acid')
plt.ylabel('Frequency Difference (Soluble - Insoluble)')
plt.title('Amino Acid Composition: Soluble vs Insoluble Proteins')
plt.show()

## 5. Group-by Operations

In [ ]:
# Statistics by organism
organism_stats = df.groupby('organism').agg({
    'length': ['mean', 'std', 'min', 'max'],
    'soluble': 'mean',  # Fraction soluble
    'thermal_stability': 'mean',
    'protein_id': 'count'
}).round(2)

organism_stats.columns = ['_'.join(col) for col in organism_stats.columns]
organism_stats = organism_stats.rename(columns={'protein_id_count': 'n_proteins'})
organism_stats

In [ ]:
# Create length bins
df['length_bin'] = pd.cut(df['length'], bins=[0, 100, 200, 300, 500], 
                          labels=['<100', '100-200', '200-300', '300-500'])

# Solubility by length bin
length_solubility = df.groupby('length_bin')['soluble'].agg(['mean', 'count'])
length_solubility.columns = ['solubility_rate', 'count']
length_solubility

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
length_solubility['solubility_rate'].plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)
ax.set_xlabel('Sequence Length Bin')
ax.set_ylabel('Solubility Rate')
ax.set_title('Solubility Rate by Protein Length')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.axhline(y=df['soluble'].mean(), color='red', linestyle='--', label='Overall average')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Data Splitting for Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split

# Standard random split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['soluble'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['soluble'])

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

# Check class balance is preserved
print(f"\nSolubility rates:")
print(f"  Train: {train_df['soluble'].mean():.3f}")
print(f"  Val:   {val_df['soluble'].mean():.3f}")
print(f"  Test:  {test_df['soluble'].mean():.3f}")

In [ ]:
# Split by organism (to avoid data leakage from organism-specific patterns)
def split_by_group(df, group_col, test_frac=0.2):
    """
    Split dataset ensuring groups don't overlap between train and test.
    """
    groups = df[group_col].unique()
    np.random.shuffle(groups)
    
    n_test = int(len(groups) * test_frac)
    test_groups = set(groups[:n_test])
    
    train_mask = ~df[group_col].isin(test_groups)
    return df[train_mask].copy(), df[~train_mask].copy()

train_by_org, test_by_org = split_by_group(df, 'organism', test_frac=0.25)

print(f"Train organisms: {train_by_org['organism'].unique()}")
print(f"Test organisms: {test_by_org['organism'].unique()}")
print(f"\nTrain size: {len(train_by_org)}, Test size: {len(test_by_org)}")

## 7. Saving and Loading Data

In [ ]:
# Save to CSV
# train_df.to_csv('train_proteins.csv', index=False)

# Save to parquet (more efficient for large datasets)
# train_df.to_parquet('train_proteins.parquet', index=False)

# Save only sequences to FASTA-like format
def save_to_fasta(df, filepath, id_col='protein_id', seq_col='sequence'):
    """Save sequences to FASTA format."""
    lines = []
    for _, row in df.iterrows():
        lines.append(f">{row[id_col]}")
        lines.append(row[seq_col])
    
    with open(filepath, 'w') as f:
        f.write('\n'.join(lines))
    print(f"Saved {len(df)} sequences to {filepath}")

# Example (commented to avoid file creation)
# save_to_fasta(train_df, 'train_sequences.fasta')

## Summary

In this notebook, we covered:

1. **Creating DataFrames** from protein data
2. **Exploration** using describe(), value_counts(), groupby()
3. **Filtering** with boolean indexing and query()
4. **Feature engineering** - computing amino acid composition
5. **Aggregation** with groupby for statistics by group
6. **Data splitting** for machine learning (random vs group-based)

Pandas is essential for preprocessing protein datasets before feeding them to ML models.